In [1]:
import duckdb
import pandas

In [2]:
sql_query = '''
show tables
'''

with duckdb.connect('data/chicago_crimes.db') as con:
    display(con.sql(sql_query).df())

,name
0,chicago_crimes
1,gold_crimes_by_district
2,gold_crimes_by_month_type
3,silver_crimes
4,silver_crimes_type


In [3]:
CSV = "data/raw/chicago_crimes_2026.csv"

with duckdb.connect("data/chicago_crimes.db") as con:
    con.sql(f"""
        create or replace table chicago_crimes as
        select *
        from read_csv_auto('{CSV}', header = true)
    """)
    con.sql("select * as filas from chicago_crimes").show()

┌──────────┬─────────────┬─────────────────────┬───┬──────────────┬───────────────┬──────────────────────┐
│    ID    │ Case Number │        Date         │ … │   Latitude   │   Longitude   │       Location       │
│  int64   │   varchar   │      timestamp      │   │    double    │    double     │       varchar        │
├──────────┼─────────────┼─────────────────────┼───┼──────────────┼───────────────┼──────────────────────┤
│ 14351522 │ JK442283    │ 2026-10-06 00:00:00 │ … │ 41.887481203 │ -87.645372811 │ POINT (-87.6453728…  │
│ 14350574 │ JK440996    │ 2026-10-06 00:00:00 │ … │ 41.966047906 │ -87.661060061 │ POINT (-87.6610600…  │
│ 14354393 │ JK445985    │ 2026-10-06 00:00:00 │ … │ 41.929417949 │  -87.70612448 │ POINT (-87.7061244…  │
│ 14349985 │ JK440553    │ 2026-10-06 00:00:00 │ … │ 41.892017399 │ -87.624165306 │ POINT (-87.6241653…  │
│ 14350659 │ JK441321    │ 2026-10-06 00:00:00 │ … │ 41.692833841 │  -87.60431945 │ POINT (-87.6043194…  │
│ 14349733 │ JK440247    │ 2026-10-06

In [4]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("select * from gold_crimes_by_month_type order by 1, 3 desc").df()

df.head(10)

,month,primary_type,total_crimes,total_arrests,arrest_rate
0,2026-01-01,THEFT,3701,385.0,0.104
1,2026-01-01,BATTERY,3027,651.0,0.215
2,2026-01-01,CRIMINAL DAMAGE,1820,80.0,0.044
3,2026-01-01,ASSAULT,1438,204.0,0.142
4,2026-01-01,MOTOR VEHICLE THEFT,1334,44.0,0.033
5,2026-01-01,OTHER OFFENSE,1253,219.0,0.175
6,2026-01-01,DECEPTIVE PRACTICE,1236,37.0,0.030
7,2026-01-01,BURGLARY,749,37.0,0.049
8,2026-01-01,NARCOTICS,577,553.0,0.958
9,2026-01-01,CRIMINAL TRESPASS,456,145.0,0.318


In [5]:
con = duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True)
con.sql("show all tables").show()
con.sql("describe chicago_crimes").show()

┌────────────────┬─────────┬──────────────────────┬──────────────────────┬─────────────────────────────────┬───────────┐
│    database    │ schema  │         name         │     column_names     │          column_types           │ temporary │
│    varchar     │ varchar │       varchar        │      varchar[]       │            varchar[]            │  boolean  │
├────────────────┼─────────┼──────────────────────┼──────────────────────┼─────────────────────────────────┼───────────┤
│ chicago_crimes │ main    │ chicago_crimes       │ [ID, Case Number, …  │ [BIGINT, VARCHAR, TIMESTAMP, …  │ false     │
│ chicago_crimes │ main    │ gold_crimes_by_dis…  │ [district, total_c…  │ [VARCHAR, BIGINT, INTEGER, DO…  │ false     │
│ chicago_crimes │ main    │ gold_crimes_by_mon…  │ [month, primary_ty…  │ [DATE, VARCHAR, BIGINT, HUGEI…  │ false     │
│ chicago_crimes │ main    │ silver_crimes        │ [crime_id, case_nu…  │ [BIGINT, VARCHAR, TIMESTAMP, …  │ false     │
│ chicago_crimes │ main    │ sil

In [6]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("""
        select * from gold_crimes_by_month_type
        order by month, total_crimes desc
    """).df()

df.head(100)

,month,primary_type,total_crimes,total_arrests,arrest_rate
0,2026-01-01,THEFT,3701,385.0,0.104
1,2026-01-01,BATTERY,3027,651.0,0.215
2,2026-01-01,CRIMINAL DAMAGE,1820,80.0,0.044
3,2026-01-01,ASSAULT,1438,204.0,0.142
4,2026-01-01,MOTOR VEHICLE THEFT,1334,44.0,0.033
...,...,...,...,...,...
95,2026-04-01,ROBBERY,327,29.0,0.089
96,2026-04-01,OFFENSE INVOLVING CHILDREN,147,9.0,0.061
97,2026-04-01,SEX OFFENSE,139,9.0,0.065
98,2026-04-01,CRIMINAL SEXUAL ASSAULT,133,4.0,0.030


In [7]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("""
        select * from silver_crimes_type
    """).df()

df.head(100)

,iucr,primary_type,description,fbi_code
0,0486,BATTERY,DOMESTIC BATTERY SIMPLE,08B
1,1152,DECEPTIVE PRACTICE,ILLEGAL USE CASH CARD,11
2,0920,MOTOR VEHICLE THEFT,ATTEMPT - AUTOMOBILE,07
3,2025,NARCOTICS,POSSESS - HALLUCINOGENS,18
4,2870,PUBLIC PEACE VIOLATION,PEEPING TOM,24
...,...,...,...,...
95,1199,DECEPTIVE PRACTICE,"PUBLIC AIR WIRE AND MAIL FRAUD - VIA WIRE, RAD...",11
96,2896,PUBLIC PEACE VIOLATION,TIRE DEFLATION DEVICE DEPLOYMENT,24
97,1791,OFFENSE INVOLVING CHILDREN,HARBOR RUNAWAY,20
98,2160,NARCOTICS,SALE / DELIVER - DRUG PARAPHERNALIA,26


In [8]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("""
        select * from silver_crimes where location_description='STREET'
    """).df()

df.head(100)

,crime_id,case_number,occurred_at,block,iucr,primary_type,description,location_description,is_arrest,is_domestic,beat,district,ward,community_area,fbi_code,latitude,longitude,updated_on
0,14351522,JK442283,2026-10-06 00:00:00,006XX W WAYMAN ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,1214,012,34,28,14,41.887481,-87.645373,2026-10-13 15:47:24
1,14354393,JK445985,2026-10-06 00:00:00,026XX N TROY ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,1411,014,35,22,14,41.929418,-87.706124,2026-10-13 15:47:24
2,14349733,JK440247,2026-10-06 00:00:00,067XX N GREENVIEW AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,False,False,2432,024,49,1,06,42.004137,-87.668128,2026-10-13 15:47:24
3,14349833,JK440479,2026-10-05 23:30:00,005XX S DAMEN AVE,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,1225,012,28,28,07,41.874716,-87.676478,2026-10-13 15:47:24
4,14349653,JK440102,2026-10-05 23:20:00,0000X E 112TH PL,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,0531,005,9,49,07,41.689873,-87.622377,2026-10-13 15:47:24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,14349299,JK439816,2026-10-05 13:55:00,085XX S ROCKWELL ST,0460,BATTERY,SIMPLE,STREET,False,False,0835,008,18,70,08B,41.737527,-87.688125,2026-10-13 15:47:24
96,14349045,JK439539,2026-10-05 13:50:00,038XX W ROOSEVELT RD,5002,OTHER OFFENSE,OTHER VEHICLE OFFENSE,STREET,False,False,1011,010,24,29,26,41.866235,-87.720517,2026-10-13 15:47:24
97,14349369,JK439717,2026-10-05 13:40:00,0000X W OHIO ST,0710,THEFT,THEFT FROM MOTOR VEHICLE,STREET,False,False,1832,018,42,8,06,41.892441,-87.630147,2026-10-13 15:47:24
98,14349388,JK439966,2026-10-05 13:35:00,004XX N LA SALLE DR,0460,BATTERY,SIMPLE,STREET,False,False,1831,018,42,8,08B,41.889393,-87.632535,2026-10-13 15:47:24
